In [0]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Data Preparation for Modelling

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report,
                             RocCurveDisplay, PrecisionRecallDisplay)
import time

OUTPUT_PATH_figures = "/Users/chamika/Desktop/git/machine_learning/Question_1/outputs/figures/"
DATA_PATH_raw    = "/Users/chamika/Desktop/git/machine_learning/Question_1/data/raw/"
DATA_PATH_processed    = "/Users/chamika/Desktop/git/machine_learning/Question_1/data/processed/"

In [0]:
# Cell 24: Prepare data for model training
print("=" * 60)
print("DATA PREPARATION FOR MODELLING")
print("=" * 60)

# Load final dataset
df_model = pd.read_csv(DATA_PATH_processed + 'df_final.csv')
print(f"✅ Loaded df_final: {df_model.shape}")

# Separate features and target
X = df_model.drop(columns=['TransactionID', 'isFraud'])
y = df_model['isFraud']

print(f"\nFeatures (X): {X.shape}")
print(f"Target (y):   {y.shape}")
print(f"Fraud rate:   {y.mean()*100:.2f}%")

# Train/test split — stratified to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # preserves 3.5% fraud in both sets
)

print(f"\n✅ Train/Test Split (80/20 stratified):")
print(f"   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   Train fraud rate: {y_train.mean()*100:.2f}%")
print(f"   Test fraud rate:  {y_test.mean()*100:.2f}%")

# Scale features — needed for Logistic Regression and KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\n✅ Features scaled (StandardScaler)")
print(f"   Mean ≈ 0, Std ≈ 1 for all features")

# Class imbalance ratio — needed for XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\n✅ Class imbalance ratio: {scale_pos_weight:.1f}:1")
print(f"   (Used as scale_pos_weight in XGBoost)")


# Model 1: Logistic Regression

In [0]:
# Cell 25: Model 1 — Logistic Regression (Linear Baseline)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report,
                             RocCurveDisplay, PrecisionRecallDisplay)
import time

print("=" * 60)
print("MODEL 1: LOGISTIC REGRESSION")
print("=" * 60)

# ── Train ─────────────────────────────────────────────────────
start = time.time()

lr_model = LogisticRegression(
    class_weight='balanced',   # handles 27.6:1 imbalance
    max_iter=1000,             # enough iterations to converge
    random_state=42,
    C=0.1                      # L2 regularisation — controls overfitting
)

lr_model.fit(X_train_scaled, y_train)
train_time = time.time() - start
print(f"\n✅ Training complete ({train_time:.1f}s)")

# ── Predict ───────────────────────────────────────────────────
y_pred_lr    = lr_model.predict(X_test_scaled)
y_proba_lr   = lr_model.predict_proba(X_test_scaled)[:, 1]

# ── Metrics ───────────────────────────────────────────────────
acc_lr    = accuracy_score(y_test, y_pred_lr)
prec_lr   = precision_score(y_test, y_pred_lr)
rec_lr    = recall_score(y_test, y_pred_lr)
f1_lr     = f1_score(y_test, y_pred_lr)
roc_lr    = roc_auc_score(y_test, y_proba_lr)
pr_lr     = average_precision_score(y_test, y_proba_lr)

print(f"\n📊 Performance Metrics:")
print(f"   Accuracy:  {acc_lr:.4f}")
print(f"   Precision: {prec_lr:.4f}")
print(f"   Recall:    {rec_lr:.4f}")
print(f"   F1 Score:  {f1_lr:.4f}")
print(f"   ROC-AUC:   {roc_lr:.4f}")
print(f"   PR-AUC:    {pr_lr:.4f}")
print(f"   Train time:{train_time:.1f}s")

# ── Confusion Matrix ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Confusion matrix
cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
axes[0].set_title('Confusion Matrix\nLogistic Regression',
                   fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Annotate TN/FP/FN/TP
tn, fp, fn, tp = cm.ravel()
print(f"\n📊 Confusion Matrix:")
print(f"   True Negatives  (TN): {tn:,}  — Legitimate correctly identified")
print(f"   False Positives (FP): {fp:,}  — Legitimate wrongly flagged as fraud")
print(f"   False Negatives (FN): {fn:,}  — Fraud missed!")
print(f"   True Positives  (TP): {tp:,}  — Fraud correctly caught")

# Plot 2: ROC Curve
# New
RocCurveDisplay.from_predictions(
    y_test, y_proba_lr, ax=axes[1],
    name=f'Logistic Regression (AUC={roc_lr:.3f})',
    curve_kwargs={'color': '#3498db'})
axes[1].plot([0,1],[0,1],'k--', linewidth=1, label='Random classifier')
axes[1].set_title('ROC Curve\nLogistic Regression',
                   fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

# Plot 3: Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba_lr, ax=axes[2],
    name=f'Logistic Regression (PR-AUC={pr_lr:.3f})',
    curve_kwargs={'color': '#e74c3c'})
axes[2].axhline(y=y_test.mean(), color='k', linestyle='--',
                linewidth=1, label=f'Baseline ({y_test.mean():.3f})')
axes[2].set_title('Precision-Recall Curve\nLogistic Regression',
                   fontsize=12, fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Model 1: Logistic Regression — Evaluation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'model1_logistic_regression.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred_lr,
      target_names=['Legitimate', 'Fraud']))

# Store results for comparison table later
results = {}
results['Logistic Regression'] = {
    'Accuracy': acc_lr, 'Precision': prec_lr,
    'Recall': rec_lr, 'F1': f1_lr,
    'ROC-AUC': roc_lr, 'PR-AUC': pr_lr,
    'Train Time (s)': round(train_time, 1)
}
print("\n✅ Results stored for comparison table")

# Model 2: KNN

In [0]:
# Cell 26: Model 2 — KNN (Distance-based)
from sklearn.neighbors import KNeighborsClassifier
import time

print("=" * 60)
print("MODEL 2: K-NEAREST NEIGHBOURS (KNN)")
print("=" * 60)

# KNN is slow on large datasets — use a stratified sample
print("Sampling 50,000 rows for KNN (computational constraint)...")
from sklearn.model_selection import train_test_split

# Sample from training set — stratified
X_train_knn, _, y_train_knn, _ = train_test_split(
    X_train_scaled, y_train,
    train_size=50000,
    random_state=42,
    stratify=y_train
)

print(f"KNN training sample: {X_train_knn.shape}")
print(f"Fraud rate in sample: {y_train_knn.mean()*100:.2f}%")

# ── Train ─────────────────────────────────────────────────────
start = time.time()

knn_model = KNeighborsClassifier(
    n_neighbors=7,        # odd number avoids ties
    metric='euclidean',
    weights='distance',   # closer neighbours weighted more
    n_jobs=-1
)

knn_model.fit(X_train_knn, y_train_knn)
train_time = time.time() - start
print(f"\n✅ Training complete ({train_time:.1f}s)")

# ── Predict ───────────────────────────────────────────────────
print("Predicting on test set (may take 2-3 minutes)...")
start_pred = time.time()
y_pred_knn  = knn_model.predict(X_test_scaled)
y_proba_knn = knn_model.predict_proba(X_test_scaled)[:, 1]
pred_time = time.time() - start_pred
print(f"✅ Prediction complete ({pred_time:.1f}s)")

# ── Metrics ───────────────────────────────────────────────────
acc_knn  = accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn)
rec_knn  = recall_score(y_test, y_pred_knn)
f1_knn   = f1_score(y_test, y_pred_knn)
roc_knn  = roc_auc_score(y_test, y_proba_knn)
pr_knn   = average_precision_score(y_test, y_proba_knn)

print(f"\n📊 Performance Metrics:")
print(f"   Accuracy:  {acc_knn:.4f}")
print(f"   Precision: {prec_knn:.4f}")
print(f"   Recall:    {rec_knn:.4f}")
print(f"   F1 Score:  {f1_knn:.4f}")
print(f"   ROC-AUC:   {roc_knn:.4f}")
print(f"   PR-AUC:    {pr_knn:.4f}")

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Confusion matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
axes[0].set_title('Confusion Matrix\nKNN', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

tn, fp, fn, tp = cm_knn.ravel()
print(f"\n📊 Confusion Matrix:")
print(f"   True Negatives  (TN): {tn:,}  — Legitimate correctly identified")
print(f"   False Positives (FP): {fp:,}  — Legitimate wrongly flagged")
print(f"   False Negatives (FN): {fn:,}  — Fraud missed!")
print(f"   True Positives  (TP): {tp:,}  — Fraud correctly caught")

# Plot 2: ROC Curve
RocCurveDisplay.from_predictions(
    y_test, y_proba_knn, ax=axes[1],
    name=f'KNN (AUC={roc_knn:.3f})',
    curve_kwargs={'color': '#9b59b6'})
axes[1].plot([0,1],[0,1],'k--', linewidth=1, label='Random classifier')
axes[1].set_title('ROC Curve\nKNN', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

# Plot 3: Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba_knn, ax=axes[2],
    name=f'KNN (PR-AUC={pr_knn:.3f})',
    curve_kwargs={'color': '#9b59b6'})
axes[2].axhline(y=y_test.mean(), color='k', linestyle='--',
                linewidth=1, label=f'Baseline ({y_test.mean():.3f})')
axes[2].set_title('Precision-Recall Curve\nKNN',
                   fontsize=12, fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Model 2: KNN — Evaluation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'model2_knn.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred_knn,
      target_names=['Legitimate', 'Fraud']))

# Store results
results['KNN'] = {
    'Accuracy': acc_knn, 'Precision': prec_knn,
    'Recall': rec_knn, 'F1': f1_knn,
    'ROC-AUC': roc_knn, 'PR-AUC': pr_knn,
    'Train Time (s)': round(train_time, 1)
}
print("✅ Results stored for comparison table")

# Model 3: Random Forest

In [0]:
# Cell 27: Model 3 — Random Forest (Tree-based)
from sklearn.ensemble import RandomForestClassifier
import time

print("=" * 60)
print("MODEL 3: RANDOM FOREST")
print("=" * 60)

# ── Train ─────────────────────────────────────────────────────
start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,          # 100 trees
    max_depth=15,              # prevent overfitting
    min_samples_leaf=10,       # minimum samples per leaf
    class_weight='balanced',   # handles 27.6:1 imbalance
    random_state=42
)

rf_model.fit(X_train, y_train)
train_time = time.time() - start
print(f"\n✅ Training complete ({train_time:.1f}s)")

# ── Predict ───────────────────────────────────────────────────
y_pred_rf  = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# ── Metrics ───────────────────────────────────────────────────
acc_rf  = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf  = recall_score(y_test, y_pred_rf)
f1_rf   = f1_score(y_test, y_pred_rf)
roc_rf  = roc_auc_score(y_test, y_proba_rf)
pr_rf   = average_precision_score(y_test, y_proba_rf)

print(f"\n📊 Performance Metrics:")
print(f"   Accuracy:  {acc_rf:.4f}")
print(f"   Precision: {prec_rf:.4f}")
print(f"   Recall:    {rec_rf:.4f}")
print(f"   F1 Score:  {f1_rf:.4f}")
print(f"   ROC-AUC:   {roc_rf:.4f}")
print(f"   PR-AUC:    {pr_rf:.4f}")
print(f"   Train time:{train_time:.1f}s")

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
axes[0].set_title('Confusion Matrix\nRandom Forest',
                   fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

tn, fp, fn, tp = cm_rf.ravel()
print(f"\n📊 Confusion Matrix:")
print(f"   True Negatives  (TN): {tn:,}  — Legitimate correctly identified")
print(f"   False Positives (FP): {fp:,}  — Legitimate wrongly flagged")
print(f"   False Negatives (FN): {fn:,}  — Fraud missed!")
print(f"   True Positives  (TP): {tp:,}  — Fraud correctly caught")

# Plot 2: ROC Curve
RocCurveDisplay.from_predictions(
    y_test, y_proba_rf, ax=axes[1],
    name=f'Random Forest (AUC={roc_rf:.3f})',
    curve_kwargs={'color': '#2ecc71'})
axes[1].plot([0,1],[0,1],'k--', linewidth=1, label='Random classifier')
axes[1].set_title('ROC Curve\nRandom Forest',
                   fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

# Plot 3: Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba_rf, ax=axes[2],
    name=f'Random Forest (PR-AUC={pr_rf:.3f})',
    curve_kwargs={'color': '#2ecc71'})
axes[2].axhline(y=y_test.mean(), color='k', linestyle='--',
                linewidth=1, label=f'Baseline ({y_test.mean():.3f})')
axes[2].set_title('Precision-Recall Curve\nRandom Forest',
                   fontsize=12, fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Model 3: Random Forest — Evaluation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'model3_random_forest.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred_rf,
      target_names=['Legitimate', 'Fraud']))

# Store results
results['Random Forest'] = {
    'Accuracy': acc_rf, 'Precision': prec_rf,
    'Recall': rec_rf, 'F1': f1_rf,
    'ROC-AUC': roc_rf, 'PR-AUC': pr_rf,
    'Train Time (s)': round(train_time, 1)
}
print("✅ Results stored for comparison table")

# Model 4: XGBoost

In [0]:
# Cell 28: Model 4 — XGBoost (Ensemble Boosting)
from xgboost import XGBClassifier
import time

print("=" * 60)
print("MODEL 4: XGBOOST")
print("=" * 60)

# ── Train ─────────────────────────────────────────────────────
start = time.time()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # 27.6:1 imbalance
    eval_metric='aucpr',                # optimise for PR-AUC
    random_state=42,
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

train_time = time.time() - start
print(f"\n✅ Training complete ({train_time:.1f}s)")

# ── Predict ───────────────────────────────────────────────────
y_pred_xgb  = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# ── Metrics ───────────────────────────────────────────────────
acc_xgb  = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb  = recall_score(y_test, y_pred_xgb)
f1_xgb   = f1_score(y_test, y_pred_xgb)
roc_xgb  = roc_auc_score(y_test, y_proba_xgb)
pr_xgb   = average_precision_score(y_test, y_proba_xgb)

print(f"\n📊 Performance Metrics:")
print(f"   Accuracy:  {acc_xgb:.4f}")
print(f"   Precision: {prec_xgb:.4f}")
print(f"   Recall:    {rec_xgb:.4f}")
print(f"   F1 Score:  {f1_xgb:.4f}")
print(f"   ROC-AUC:   {roc_xgb:.4f}")
print(f"   PR-AUC:    {pr_xgb:.4f}")
print(f"   Train time:{train_time:.1f}s")

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
axes[0].set_title('Confusion Matrix\nXGBoost',
                   fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

tn, fp, fn, tp = cm_xgb.ravel()
print(f"\n📊 Confusion Matrix:")
print(f"   True Negatives  (TN): {tn:,}  — Legitimate correctly identified")
print(f"   False Positives (FP): {fp:,}  — Legitimate wrongly flagged")
print(f"   False Negatives (FN): {fn:,}  — Fraud missed!")
print(f"   True Positives  (TP): {tp:,}  — Fraud correctly caught")

# Plot 2: ROC Curve
RocCurveDisplay.from_predictions(
    y_test, y_proba_xgb, ax=axes[1],
    name=f'XGBoost (AUC={roc_xgb:.3f})',
    curve_kwargs={'color': '#e67e22'})
axes[1].plot([0,1],[0,1],'k--', linewidth=1, label='Random classifier')
axes[1].set_title('ROC Curve\nXGBoost',
                   fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

# Plot 3: Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba_xgb, ax=axes[2],
    name=f'XGBoost (PR-AUC={pr_xgb:.3f})',
    curve_kwargs={'color': '#e67e22'})
axes[2].axhline(y=y_test.mean(), color='k', linestyle='--',
                linewidth=1, label=f'Baseline ({y_test.mean():.3f})')
axes[2].set_title('Precision-Recall Curve\nXGBoost',
                   fontsize=12, fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Model 4: XGBoost — Evaluation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'model4_xgboost.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred_xgb,
      target_names=['Legitimate', 'Fraud']))

# Store results
results['XGBoost'] = {
    'Accuracy': acc_xgb, 'Precision': prec_xgb,
    'Recall': rec_xgb, 'F1': f1_xgb,
    'ROC-AUC': roc_xgb, 'PR-AUC': pr_xgb,
    'Train Time (s)': round(train_time, 1)
}
print("✅ Results stored for comparison table")

# Model 5: ANN (MLPClassifier)

In [0]:
# Cell 29: Model 5 — ANN (Multi-Layer Perceptron)
# Using scikit-learn MLPClassifier — fully valid ANN implementation
# Compatible with Python 3.12 + Apple Silicon M3
from sklearn.neural_network import MLPClassifier
import time

print("=" * 60)
print("MODEL 5: ARTIFICIAL NEURAL NETWORK (ANN)")
print("Architecture: 3-layer MLP [256 → 128 → 64 → 1]")
print("=" * 60)

# ── Train ─────────────────────────────────────────────────────
start = time.time()

ann_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),  # 3 hidden layers
    activation='relu',                   # ReLU activation
    solver='adam',                       # Adam optimiser
    alpha=0.001,                         # L2 regularisation
    batch_size=2048,                     # mini-batch size
    learning_rate='adaptive',            # reduces LR on plateau
    learning_rate_init=0.001,
    max_iter=50,                         # epochs
    early_stopping=True,                 # stop if no improvement
    validation_fraction=0.1,             # 10% for validation
    n_iter_no_change=5,                  # patience
    random_state=42,
    verbose=True
)

ann_model.fit(X_train_scaled, y_train)
train_time = time.time() - start
print(f"\n✅ Training complete ({train_time:.1f}s)")
print(f"   Epochs run: {ann_model.n_iter_}")

# ── Predict ───────────────────────────────────────────────────
y_proba_ann = ann_model.predict_proba(X_test_scaled)[:, 1]
y_pred_ann  = ann_model.predict(X_test_scaled)

# ── Metrics ───────────────────────────────────────────────────
acc_ann  = accuracy_score(y_test, y_pred_ann)
prec_ann = precision_score(y_test, y_pred_ann)
rec_ann  = recall_score(y_test, y_pred_ann)
f1_ann   = f1_score(y_test, y_pred_ann)
roc_ann  = roc_auc_score(y_test, y_proba_ann)
pr_ann   = average_precision_score(y_test, y_proba_ann)

print(f"\n📊 Performance Metrics:")
print(f"   Accuracy:  {acc_ann:.4f}")
print(f"   Precision: {prec_ann:.4f}")
print(f"   Recall:    {rec_ann:.4f}")
print(f"   F1 Score:  {f1_ann:.4f}")
print(f"   ROC-AUC:   {roc_ann:.4f}")
print(f"   PR-AUC:    {pr_ann:.4f}")
print(f"   Train time:{train_time:.1f}s")

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Confusion matrix
cm_ann = confusion_matrix(y_test, y_pred_ann)
sns.heatmap(cm_ann, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
axes[0].set_title('Confusion Matrix\nANN (MLP)',
                   fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

tn, fp, fn, tp = cm_ann.ravel()
print(f"\n📊 Confusion Matrix:")
print(f"   True Negatives  (TN): {tn:,}  — Legitimate correctly identified")
print(f"   False Positives (FP): {fp:,}  — Legitimate wrongly flagged")
print(f"   False Negatives (FN): {fn:,}  — Fraud missed!")
print(f"   True Positives  (TP): {tp:,}  — Fraud correctly caught")

# Plot 2: ROC Curve
RocCurveDisplay.from_predictions(
    y_test, y_proba_ann, ax=axes[1],
    name=f'ANN (AUC={roc_ann:.3f})',
    curve_kwargs={'color': '#e74c3c'})
axes[1].plot([0,1],[0,1],'k--', linewidth=1, label='Random classifier')
axes[1].set_title('ROC Curve\nANN (MLP)',
                   fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

# Plot 3: Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba_ann, ax=axes[2],
    name=f'ANN (PR-AUC={pr_ann:.3f})',
    curve_kwargs={'color': '#e74c3c'})
axes[2].axhline(y=y_test.mean(), color='k', linestyle='--',
                linewidth=1, label=f'Baseline ({y_test.mean():.3f})')
axes[2].set_title('Precision-Recall Curve\nANN (MLP)',
                   fontsize=12, fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Model 5: ANN (MLP) — Evaluation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'model5_ann.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Loss curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ann_model.loss_curve_, color='#e74c3c', linewidth=2, label='Train Loss')
if ann_model.best_loss_ is not None:
    ax.axhline(y=ann_model.best_loss_, color='blue', linestyle='--',
               linewidth=1.5, label=f'Best loss: {ann_model.best_loss_:.4f}')
ax.set_title('ANN Training Loss Curve', fontsize=13, fontweight='bold')
ax.set_xlabel('Iterations')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'model5_ann_loss.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred_ann,
      target_names=['Legitimate', 'Fraud']))

# Store results
results['ANN'] = {
    'Accuracy': acc_ann, 'Precision': prec_ann,
    'Recall': rec_ann, 'F1': f1_ann,
    'ROC-AUC': roc_ann, 'PR-AUC': pr_ann,
    'Train Time (s)': round(train_time, 1)
}
print("✅ Results stored for comparison table")

# Model Comparison Table & Visualisation

In [0]:
# Cell 30: Model comparison table and visualisation
import pandas as pd

print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)

# Build comparison dataframe
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

# Rank by PR-AUC (most relevant for imbalanced data)
results_df = results_df.sort_values('PR-AUC', ascending=False)

print("\n📊 Model Comparison Table (sorted by PR-AUC):")
print(results_df.to_string())

# ── Visualisation ─────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

colors = ['#3498db', '#9b59b6', '#2ecc71', '#e67e22', '#e74c3c']
models = results_df.index.tolist()

for idx, metric in enumerate(metrics):
    values = results_df[metric].values
    bars = axes[idx].bar(models, values, color=colors,
                          edgecolor='black', linewidth=0.5)
    axes[idx].set_title(f'{metric}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(metric)
    axes[idx].set_ylim(0, min(1.1, max(values) * 1.15))
    axes[idx].tick_params(axis='x', rotation=30)

    # Highlight best
    best_idx = values.argmax()
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

    # Value labels
    for bar, val in zip(bars, values):
        axes[idx].text(bar.get_x() + bar.get_width()/2.,
                        bar.get_height() + 0.01,
                        f'{val:.3f}', ha='center',
                        va='bottom', fontsize=8, fontweight='bold')

plt.suptitle('Model Comparison — All 5 Models\n(Gold border = best per metric)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'model_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("\n🏆 Best model per metric:")
for metric in metrics:
    best = results_df[metric].idxmax()
    val  = results_df[metric].max()
    print(f"   {metric:12s}: {best:25s} ({val:.4f})")

## Hyperparameter optimisation was applied to the two best-performing models (XGBoost and ANN) using RandomizedSearchCV, as exhaustive grid search across all five models would be computationally prohibitive on a dataset of 590,540 instances.

# Cross-Validation

In [0]:
# Cell 31: Cross-validation — Stratified K-Fold on all 5 models
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("CROSS-VALIDATION — STRATIFIED 5-FOLD")
print("=" * 60)
print("Using stratified folds to preserve 3.5% fraud rate in each fold")
print("Scoring on: F1, ROC-AUC, PR-AUC\n")

# ── Setup ─────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Use a sample for speed — full 590k would take hours for KNN/ANN
# 100k sample is statistically robust and academically justified
sample_idx = pd.Series(range(len(X))).sample(n=100000, random_state=42)
X_cv = X.iloc[sample_idx].reset_index(drop=True)
y_cv = y.iloc[sample_idx].reset_index(drop=True)

# Scale for LR, KNN, ANN
from sklearn.preprocessing import StandardScaler
scaler_cv = StandardScaler()
X_cv_scaled = scaler_cv.fit_transform(X_cv)

print(f"CV sample size: {len(X_cv):,} rows")
print(f"Fraud rate in sample: {y_cv.mean()*100:.2f}%")
print(f"Folds: 5\n")

scoring = ['f1', 'roc_auc', 'average_precision']

# ── Models to evaluate ────────────────────────────────────────
cv_models = {
    'Logistic Regression': (LogisticRegression(
        class_weight='balanced', max_iter=1000,
        random_state=42, C=0.1), X_cv_scaled),
    
    'KNN': (KNeighborsClassifier(
        n_neighbors=7, metric='euclidean',
        weights='distance', n_jobs=-1), X_cv_scaled),
    
    'Random Forest': (RandomForestClassifier(
        n_estimators=100, max_depth=15,
        min_samples_leaf=10, class_weight='balanced',
        random_state=42, n_jobs=-1), X_cv),
    
    'XGBoost': (XGBClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='aucpr', random_state=42,
        n_jobs=-1, verbosity=0), X_cv),
    
    'ANN': (MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation='relu', solver='adam',
        alpha=0.001, batch_size=2048,
        learning_rate='adaptive',
        max_iter=50, early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        random_state=42), X_cv_scaled)
}

# ── Run CV ────────────────────────────────────────────────────
cv_results = {}

for name, (model, X_data) in cv_models.items():
    print(f"Running CV for {name}...")
    start = time.time()
    
    scores = cross_validate(
        model, X_data, y_cv,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )
    
    elapsed = time.time() - start
    
    cv_results[name] = {
        'F1 Mean':       scores['test_f1'].mean(),
        'F1 Std':        scores['test_f1'].std(),
        'ROC-AUC Mean':  scores['test_roc_auc'].mean(),
        'ROC-AUC Std':   scores['test_roc_auc'].std(),
        'PR-AUC Mean':   scores['test_average_precision'].mean(),
        'PR-AUC Std':    scores['test_average_precision'].std(),
    }
    
    print(f"   F1:      {scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}")
    print(f"   ROC-AUC: {scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}")
    print(f"   PR-AUC:  {scores['test_average_precision'].mean():.4f} ± {scores['test_average_precision'].std():.4f}")
    print(f"   Time:    {elapsed:.1f}s\n")

# ── Summary table ─────────────────────────────────────────────
cv_df = pd.DataFrame(cv_results).T.round(4)
cv_df = cv_df.sort_values('PR-AUC Mean', ascending=False)

print("=" * 60)
print("CV RESULTS SUMMARY (sorted by PR-AUC Mean):")
print("=" * 60)
print(cv_df.to_string())

# ── Plot CV results ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics_cv = [
    ('F1 Mean', 'F1 Std', 'F1 Score'),
    ('ROC-AUC Mean', 'ROC-AUC Std', 'ROC-AUC'),
    ('PR-AUC Mean', 'PR-AUC Std', 'PR-AUC')
]
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#9b59b6', '#3498db']
model_names = cv_df.index.tolist()

for idx, (mean_col, std_col, title) in enumerate(metrics_cv):
    means = cv_df[mean_col].values
    stds  = cv_df[std_col].values
    
    bars = axes[idx].bar(model_names, means,
                          yerr=stds, capsize=5,
                          color=colors[:len(model_names)],
                          edgecolor='black', linewidth=0.5,
                          error_kw={'linewidth': 2})
    
    axes[idx].set_title(f'{title}\n(Mean ± Std across 5 folds)',
                         fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(title)
    axes[idx].tick_params(axis='x', rotation=30)
    axes[idx].set_ylim(0, min(1.1, max(means + stds) * 1.2))
    
    # Value labels
    for bar, mean, std in zip(bars, means, stds):
        axes[idx].text(bar.get_x() + bar.get_width()/2.,
                        bar.get_height() + std + 0.01,
                        f'{mean:.3f}', ha='center',
                        va='bottom', fontsize=8, fontweight='bold')

plt.suptitle('5-Fold Stratified Cross-Validation Results',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'cross_validation.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Cross-validation complete")

# Hyperparameter Tuning (XGBoost + ANN)

In [0]:
# Cell 32: Hyperparameter tuning — RandomizedSearchCV
# Applied to XGBoost and ANN (top 2 models by CV PR-AUC)
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("HYPERPARAMETER TUNING — RANDOMIZED SEARCH CV")
print("=" * 60)
print("Models: XGBoost + ANN (top 2 by cross-validation PR-AUC)")
print("Metric: PR-AUC (most relevant for imbalanced fraud detection)")

# Use stratified sample for tuning speed
sample_idx_tune = pd.Series(range(len(X_train))).sample(
    n=50000, random_state=42)
X_tune = X_train.iloc[sample_idx_tune].reset_index(drop=True)
y_tune = y_train.iloc[sample_idx_tune].reset_index(drop=True)

X_tune_scaled = scaler.transform(X_tune)

cv_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print(f"\nTuning sample: {len(X_tune):,} rows")
print(f"CV folds: 3 (for speed)")
print(f"Fraud rate: {y_tune.mean()*100:.2f}%")

# ══════════════════════════════════════════════════════════════
# PART 1: XGBoost Tuning
# ══════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
print("PART 1: XGBoost Hyperparameter Tuning")
print("─" * 60)

xgb_param_dist = {
    'n_estimators':    randint(100, 500),
    'max_depth':       randint(3, 10),
    'learning_rate':   uniform(0.01, 0.2),
    'subsample':       uniform(0.6, 0.4),
    'colsample_bytree':uniform(0.6, 0.4),
    'min_child_weight':randint(1, 10),
    'gamma':           uniform(0, 0.5),
}

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=20,              # 20 random combinations
    scoring='average_precision',
    cv=cv_tune,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start = time.time()
xgb_search.fit(X_tune, y_tune)
xgb_tune_time = time.time() - start

print(f"\n✅ XGBoost tuning complete ({xgb_tune_time:.1f}s)")
print(f"\n🏆 Best XGBoost parameters:")
for param, value in xgb_search.best_params_.items():
    print(f"   {param:25s}: {value}")
print(f"\n   Best CV PR-AUC: {xgb_search.best_score_:.4f}")

# Retrain best XGBoost on full training set
best_xgb = XGBClassifier(
    **xgb_search.best_params_,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
best_xgb.fit(X_train, y_train)

# Evaluate tuned XGBoost
y_pred_xgb_tuned  = best_xgb.predict(X_test)
y_proba_xgb_tuned = best_xgb.predict_proba(X_test)[:, 1]

acc_xgb_t  = accuracy_score(y_test, y_pred_xgb_tuned)
prec_xgb_t = precision_score(y_test, y_pred_xgb_tuned)
rec_xgb_t  = recall_score(y_test, y_pred_xgb_tuned)
f1_xgb_t   = f1_score(y_test, y_pred_xgb_tuned)
roc_xgb_t  = roc_auc_score(y_test, y_proba_xgb_tuned)
pr_xgb_t   = average_precision_score(y_test, y_proba_xgb_tuned)

print(f"\n📊 Tuned XGBoost vs Default XGBoost:")
print(f"   {'Metric':12s} {'Default':>10s} {'Tuned':>10s} {'Change':>10s}")
print(f"   {'─'*44}")
for metric, default, tuned in [
    ('Accuracy',  acc_xgb,  acc_xgb_t),
    ('Precision', prec_xgb, prec_xgb_t),
    ('Recall',    rec_xgb,  rec_xgb_t),
    ('F1',        f1_xgb,   f1_xgb_t),
    ('ROC-AUC',   roc_xgb,  roc_xgb_t),
    ('PR-AUC',    pr_xgb,   pr_xgb_t)]:
    change = tuned - default
    arrow  = '↑' if change > 0 else '↓'
    print(f"   {metric:12s} {default:>10.4f} {tuned:>10.4f} "
          f"{arrow}{abs(change):>8.4f}")

# ══════════════════════════════════════════════════════════════
# PART 2: ANN Tuning
# ══════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
print("PART 2: ANN Hyperparameter Tuning")
print("─" * 60)

ann_param_dist = {
    'hidden_layer_sizes': [(256,128,64), (128,64), 
                           (256,128), (512,256,128),
                           (128,64,32)],
    'alpha':              [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.0001, 0.001, 0.01],
    'batch_size':         [512, 1024, 2048],
}

ann_base = MLPClassifier(
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42
)

ann_search = RandomizedSearchCV(
    ann_base,
    param_distributions=ann_param_dist,
    n_iter=10,              # 10 combinations for ANN
    scoring='average_precision',
    cv=cv_tune,
    random_state=42,
    n_jobs=1,               # ANN can't parallelise safely
    verbose=1
)

start = time.time()
ann_search.fit(X_tune_scaled, y_tune)
ann_tune_time = time.time() - start

print(f"\n✅ ANN tuning complete ({ann_tune_time:.1f}s)")
print(f"\n🏆 Best ANN parameters:")
for param, value in ann_search.best_params_.items():
    print(f"   {param:25s}: {value}")
print(f"\n   Best CV PR-AUC: {ann_search.best_score_:.4f}")

# Retrain best ANN on full training set
best_ann = MLPClassifier(
    **ann_search.best_params_,
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42
)
best_ann.fit(X_train_scaled, y_train)

# Evaluate tuned ANN
y_pred_ann_tuned  = best_ann.predict(X_test_scaled)
y_proba_ann_tuned = best_ann.predict_proba(X_test_scaled)[:, 1]

acc_ann_t  = accuracy_score(y_test, y_pred_ann_tuned)
prec_ann_t = precision_score(y_test, y_pred_ann_tuned)
rec_ann_t  = recall_score(y_test, y_pred_ann_tuned)
f1_ann_t   = f1_score(y_test, y_pred_ann_tuned)
roc_ann_t  = roc_auc_score(y_test, y_proba_ann_tuned)
pr_ann_t   = average_precision_score(y_test, y_proba_ann_tuned)

print(f"\n📊 Tuned ANN vs Default ANN:")
print(f"   {'Metric':12s} {'Default':>10s} {'Tuned':>10s} {'Change':>10s}")
print(f"   {'─'*44}")
for metric, default, tuned in [
    ('Accuracy',  acc_ann,  acc_ann_t),
    ('Precision', prec_ann, prec_ann_t),
    ('Recall',    rec_ann,  rec_ann_t),
    ('F1',        f1_ann,   f1_ann_t),
    ('ROC-AUC',   roc_ann,  roc_ann_t),
    ('PR-AUC',    pr_ann,   pr_ann_t)]:
    change = tuned - default
    arrow  = '↑' if change > 0 else '↓'
    print(f"   {metric:12s} {default:>10.4f} {tuned:>10.4f} "
          f"{arrow}{abs(change):>8.4f}")

# ── Update results dict with tuned models ─────────────────────
results['XGBoost (Tuned)'] = {
    'Accuracy': acc_xgb_t, 'Precision': prec_xgb_t,
    'Recall': rec_xgb_t, 'F1': f1_xgb_t,
    'ROC-AUC': roc_xgb_t, 'PR-AUC': pr_xgb_t,
    'Train Time (s)': round(xgb_tune_time, 1)
}
results['ANN (Tuned)'] = {
    'Accuracy': acc_ann_t, 'Precision': prec_ann_t,
    'Recall': rec_ann_t, 'F1': f1_ann_t,
    'ROC-AUC': roc_ann_t, 'PR-AUC': pr_ann_t,
    'Train Time (s)': round(ann_tune_time, 1)
}

print(f"\n✅ Tuned models added to results table")
print(f"\n📊 Updated Results Table:")
results_df_updated = pd.DataFrame(results).T.round(4)
results_df_updated = results_df_updated.sort_values('PR-AUC', ascending=False)
print(results_df_updated.to_string())

# Final Model Ranking 
### XGBoost (Tuned) is your best model with PR-AUC 0.7875 — this becomes the selected deployment model for Sections 1.4, 1.5, and 1.6.

# Save all predictions and probabilities

In [0]:
# Save all predictions and probabilities to avoid retraining
predictions = {
    'y_test':             y_test.values,
    'y_pred_lr':          y_pred_lr,
    'y_proba_lr':         y_proba_lr,
    'y_pred_knn':         y_pred_knn,
    'y_proba_knn':        y_proba_knn,
    'y_pred_rf':          y_pred_rf,
    'y_proba_rf':         y_proba_rf,
    'y_pred_xgb':         y_pred_xgb,
    'y_proba_xgb':        y_proba_xgb,
    'y_pred_ann':         y_pred_ann,
    'y_proba_ann':        y_proba_ann,
    'y_pred_xgb_tuned':   y_pred_xgb_tuned,
    'y_proba_xgb_tuned':  y_proba_xgb_tuned,
    'y_pred_ann_tuned':   y_pred_ann_tuned,
    'y_proba_ann_tuned':  y_proba_ann_tuned,
}

pred_df = pd.DataFrame(predictions)
pred_df.to_csv(DATA_PATH_processed + 'model_predictions.csv', index=False)
print(f"✅ Predictions saved: {pred_df.shape}")

# Save results summary table
results_df_full = pd.DataFrame(results).T.round(4)
results_df_full.to_csv(DATA_PATH_processed + 'model_results.csv')
print(f"✅ Results table saved")

# Save scale_pos_weight for reference
import json
meta = {'scale_pos_weight': float(scale_pos_weight)}
with open(DATA_PATH_processed + 'model_meta.json', 'w') as f:
    json.dump(meta, f)
print(f"✅ Model metadata saved")

In [0]:
# Save tuned XGBoost model and test features for SHAP analysis
import joblib

joblib.dump(best_xgb, DATA_PATH_processed + 'best_xgb_model.pkl')
X_test.to_csv(DATA_PATH_processed + 'X_test_features.csv', index=False)

print("✅ Model and feature-named test set saved for SHAP analysis")